In [ ]:
!pip install "torch==2.10.0"
!pip install "pandas==2.2.2"
!pip install torchvision --index-url https://download.pytorch.org/whl/cpu

!pip install accelerate pillow
!pip install --upgrade transformers accelerate


In [ ]:
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
import xgrammar as xgr 

MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it")

# Load model
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Setting TokenizerInfo for XGrammar (it is need to do this just once)
tokenizer_info = xgr.TokenizerInfo.from_huggingface(processor.tokenizer)

In [ ]:
!pip install fastapi uvicorn pyngrok -q

The cell below is used to kill the process listening on port 8000, where the service (uvicorn/ngrok) runs. It is optional, but often necessary if we want to restart ngrok and start the service again on the same port; if the port is not freed, "port already in use" errors may occur. Running it stops the active process and allows you to restart the server on that port.

In [ ]:
import os
import signal
import subprocess

try:
    # Find the process ID (PID) using port 8000
    pid = int(subprocess.check_output(["lsof", "-t", "-i:8000"]).strip())
    # Kill the process
    os.kill(pid, signal.SIGKILL)
    print(print(f"✅ Successfully killed process {pid} on port 8000."))
except subprocess.CalledProcessError:
    print("ℹ️ No process was found running on port 8000. It is already free!")


In [ ]:
import subprocess, threading
from fastapi import FastAPI
from pyngrok import ngrok
import uvicorn
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessorList
import torch
from pydantic import BaseModel
import xgrammar as xgr
import json
import time

app = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str = ""
    max_tokens: int = 512
    schema: dict = None

@app.post("/generate")
async def generate(req: GenerateRequest):
    inputs = processor(text=req.prompt, return_transformers=None, return_tensors="pt").to(model.device)

    # Configure XGrammar if we receive a schema
    logits_processor_list = None
    if req.schema:
        print("Compiling XGrammar for the received schema...")
        schema_str = json.dumps(req.schema)
        compiler = xgr.GrammarCompiler(tokenizer_info)
        compiled_grammar = compiler.compile_json_schema(schema_str)
        
        # Bind XGrammar to Transformers
        xgr_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
        logits_processor_list = LogitsProcessorList([xgr_processor])
    
    with torch.no_grad():
        if logits_processor_list:
            outputs = model.generate(
                **inputs, 
                max_new_tokens=req.max_tokens,
                logits_processor=logits_processor_list
            )
        else:
            outputs = model.generate(**inputs, max_new_tokens=req.max_tokens)
    
    # Trim the prompt and decode only the new tokens
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][prompt_length:]
    
    # Decode only the new tokens
    text = processor.decode(new_tokens, skip_special_tokens=True)
    print('response: ', text)
    return {"response": text}


@app.get("/health")
async def health():
    return {"status": "ok"}

# --- Expose with ngrok ---
# You need a free token from https://ngrok.com
ngrok.set_auth_token("3ETgEA5sbb0ampjePofIgMpzwPe_5eYd1UbUfyhyZJa4ne41n")
public_url = ngrok.connect(8000)
print(f"\n🚀 Public URL: {public_url}\n")

# Start server in background
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Keep the notebook alive
import time
while True:
    time.sleep(60)